# Kraków - oferty mieszkań 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Matix732/wizualizacja-danych-uj/blob/main/Kraków_mieszkania_mapa.ipynb)

Notebook korzysta z pliku `Otodom_Flat_Listings.csv` z repozytorium GitHub.

Zakres:
- filtrujemy oferty do Krakowa,
- wyciągamy dzielnice i poddzielnice oraz cechy mieszkań,
- jeśli w ogłoszeniu brakuje lokalizacji, próbujemy znaleźć dzielnicę po nazwie ulicy,
- robimy mapę dzielnic i wizualizacje danych,

In [5]:
# Google Colab: instalacja bibliotek
%pip -q install geopandas folium osmnx geopy squarify

In [6]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import osmnx as ox
import squarify

from pathlib import Path
from matplotlib.ticker import FuncFormatter
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
print("Importy gotowe")

Importy gotowe


In [ ]:
# Wczytanie danych z repozytorium GitHub
import urllib.request

csv_url = "https://raw.githubusercontent.com/Matix732/wizualizacja-danych-uj/main/Otodom_Flat_Listings.csv"

try:
    df = pd.read_csv(csv_url)
    print(f"✓ Wczytano z GitHub: {csv_url}")
except Exception as e:
    print(f"✗ Błąd przy pobieraniu z GitHub: {e}")
    raise

print(f"Liczba rekordów (cała Polska): {len(df):,}")

# Normalizacja kolumn
rename_map = {
    "Title": "title",
    "Price": "price",
    "Location": "location",
    "Surface": "surface",
    "Number_of_Rooms": "rooms",
    "Link": "link",
    "City": "city",
}
df = df.rename(columns=rename_map)

for col in ["title", "price", "location", "surface", "rooms", "link", "city"]:
    if col not in df.columns:
        df[col] = np.nan

df["city_norm"] = df["city"].astype(str).str.lower().str.strip()
df["location_norm"] = df["location"].astype(str).str.lower().str.strip()

# Zostawiamy tylko Kraków (na podstawie city lub location)
krk = df[
    df["city_norm"].str.contains("krak", na=False)
    | df["location_norm"].str.contains("krak", na=False)
].copy()

krk["price"] = pd.to_numeric(krk["price"], errors="coerce")
krk["surface"] = pd.to_numeric(krk["surface"], errors="coerce")
krk["rooms"] = pd.to_numeric(krk["rooms"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
krk = krk.dropna(subset=["price"])
krk = krk[krk["price"].between(50_000, 10_000_000)]

print(f"Oferty w Krakowie: {len(krk):,}")
krk[["title", "price", "location", "city"]].head(5)

FileNotFoundError: Nie znaleziono pliku CSV. Umieść go w /content lub katalogu roboczym.

In [ ]:
DISTRICTS = [
    "Stare Miasto", "Grzegórzki", "Prądnik Czerwony", "Prądnik Biały", "Krowodrza",
    "Bronowice", "Zwierzyniec", "Dębniki", "Łagiewniki-Borek Fałęcki", "Swoszowice",
    "Podgórze Duchackie", "Bieżanów-Prokocim", "Podgórze", "Czyżyny", "Mistrzejowice",
    "Bieńczyce", "Wzgórza Krzesławickie", "Nowa Huta"
]

district_keywords = {
    d: [d.lower()] for d in DISTRICTS
}
district_keywords["Stare Miasto"] += ["kazimierz", "stradom", "stare miasto"]
district_keywords["Podgórze"] += ["zabłocie", "podgorze"]
district_keywords["Krowodrza"] += ["azory"]
district_keywords["Dębniki"] += ["ruczaj", "debniki"]
district_keywords["Nowa Huta"] += ["nowa huta"]

def extract_street(text: str):
    if not isinstance(text, str):
        return None
    m = re.search(r"(?:ul\.?|al\.?|os\.?)[\s]+([A-ZĄĆĘŁŃÓŚŹŻa-ząćęłńóśźż0-9\- ]{3,})", text)
    return m.group(0).strip() if m else None

def normalize_tokens(location: str):
    if not isinstance(location, str):
        return []
    return [t.strip() for t in location.split(",") if t.strip()]

def guess_district_from_text(text: str):
    if not isinstance(text, str):
        return None
    txt = text.lower()
    for district, keys in district_keywords.items():
        if any(k in txt for k in keys):
            return district
    return None

def extract_subdistrict(location: str, district: str):
    tokens = normalize_tokens(location)
    if not tokens:
        return None
    cleaned = []
    for t in tokens:
        tl = t.lower()
        if "krak" in tl or "małopol" in tl or tl in {"polska", "poland"}:
            continue
        if district and district.lower() in tl:
            continue
        cleaned.append(t)
    return cleaned[0] if cleaned else None

krk["street"] = krk["location"].apply(extract_street)
krk.loc[krk["street"].isna(), "street"] = krk.loc[krk["street"].isna(), "title"].apply(extract_street)

krk["district"] = krk["location"].apply(guess_district_from_text)
krk["subdistrict"] = krk.apply(lambda r: extract_subdistrict(r["location"], r["district"]), axis=1)

# Wymaganie 4: jeśli brak lokalizacji, użyj ulicy z tytułu i geokoduj do dzielnicy
missing_loc = krk[krk["district"].isna() & krk["street"].notna()].copy()
print(f"Brak dzielnicy po samym location: {len(missing_loc)} rekordów")

if len(missing_loc) > 0:
    geolocator = Nominatim(user_agent="krakow_district_lookup_colab")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

    def district_from_street(street):
        try:
            loc = geocode(f"{street}, Kraków, Polska", addressdetails=True)
            if not loc:
                return None, None
            rev = reverse((loc.latitude, loc.longitude), language="pl", addressdetails=True)
            addr = rev.raw.get("address", {}) if rev else {}
            district_text = " ".join([
                str(addr.get("city_district", "")),
                str(addr.get("suburb", "")),
                str(addr.get("neighbourhood", "")),
            ]).strip()
            district = guess_district_from_text(district_text)
            subdistrict = addr.get("neighbourhood") or addr.get("suburb") or addr.get("quarter")
            return district, subdistrict
        except Exception:
            return None, None

    for idx, row in missing_loc.iterrows():
        d, s = district_from_street(row["street"])
        if d:
            krk.at[idx, "district"] = d
        if s and pd.isna(krk.at[idx, "subdistrict"]):
            krk.at[idx, "subdistrict"] = s

# Domknięcie braków
krk["district"] = krk["district"].fillna("Nieustalona dzielnica")
krk["subdistrict"] = krk["subdistrict"].fillna("Nieustalona poddzielnica")
krk["price_m2"] = (krk["price"] / krk["surface"]).replace([np.inf, -np.inf], np.nan)

krk[["title", "street", "district", "subdistrict", "price", "surface"]].head(10)

In [ ]:
# Granice dzielnic Krakowa (GeoPandas + OSM)
district_gdfs = []
for district in DISTRICTS:
    try:
        q = f"{district}, Kraków, województwo małopolskie, Polska"
        g = ox.geocode_to_gdf(q)
        g["district"] = district
        district_gdfs.append(g[["district", "geometry"]])
    except Exception:
        pass

if not district_gdfs:
    raise RuntimeError("Nie udało się pobrać granic dzielnic z OSM.")

districts_gdf = pd.concat(district_gdfs, ignore_index=True)
districts_gdf = gpd.GeoDataFrame(districts_gdf, geometry="geometry", crs="EPSG:4326")
districts_gdf = districts_gdf.dissolve(by="district", as_index=False)

agg_district = (
    krk.groupby("district", dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
)

districts_plot = districts_gdf.merge(agg_district, on="district", how="left")
districts_plot[["listings", "median_price", "median_price_m2"]] = districts_plot[["listings", "median_price", "median_price_m2"]].fillna(0)

print("Dzielnice z geometrią:", len(districts_plot))
districts_plot.head(3)

In [ ]:
# Mapa dzielnic: liczba ofert (choropleth)
fig, ax = plt.subplots(1, 1, figsize=(11, 11))
districts_plot.plot(
    column="listings",
    cmap="YlOrRd",
    linewidth=1,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Brak danych"},
)

for _, row in districts_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        p = row.geometry.representative_point()
        ax.text(p.x, p.y, f"{row['district']}\n{int(row['listings'])}", fontsize=7, ha="center")

ax.set_title("Kraków - liczba ofert mieszkań wg dzielnic", fontsize=14, weight="bold")
ax.axis("off")
plt.show()

In [ ]:
# Mapa dzielnic: mediana ceny za m2 (choropleth)
fig, ax = plt.subplots(1, 1, figsize=(11, 11))
districts_plot.plot(
    column="median_price_m2",
    cmap="RdYlGn_r",
    linewidth=1,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Brak danych"},
)

for _, row in districts_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        p = row.geometry.representative_point()
        price_m2 = row['median_price_m2']
        ax.text(p.x, p.y, f"{row['district']}\n{price_m2:,.0f} PLN/m²", fontsize=7, ha="center")

ax.set_title("Kraków - mediana ceny za m² wg dzielnic", fontsize=14, weight="bold")
ax.axis("off")
plt.show()


In [ ]:
# Parallel Coordinate Plot: porównanie cen w dzielnicach
from pandas.plotting import parallel_coordinates

coords_data = (
    krk.dropna(subset=["district", "price", "surface", "price_m2"])
    .groupby("district")
    .agg({
        "price": "median",
        "surface": "median",
        "price_m2": "median",
    })
    .reset_index()
)

# Normalizacja do 0-1 dla lepszej widoczności
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
plot_data = coords_data.copy()
plot_data[["price", "surface", "price_m2"]] = scaler.fit_transform(
    plot_data[["price", "surface", "price_m2"]]
)

fig, ax = plt.subplots(figsize=(14, 6))
parallel_coordinates(plot_data, "district", ax=ax, color=sns.color_palette("tab20", len(plot_data)))
ax.set_title("Parallel Coordinates Plot: porównanie mediany ceny, powierzchni i ceny/m² wg dzielnic", fontsize=14, weight="bold")
ax.set_ylabel("Wartość (znormalizowana 0-1)")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Interaktywna mapa: najdroższe mieszkania (cena za m²)
# Wybiramy top 150 najdroższych mieszkań
krk_sorted = krk.dropna(subset=["price_m2"]).sort_values("price_m2", ascending=False)
top_expensive = krk_sorted.head(150).copy()

# Próbujemy geokodować (jeśli są współrzędne) lub używamy centroidu dzielnicy
centroids_dict = districts_plot.set_index("district").geometry.centroid.to_dict()

m_expensive = folium.Map(location=[50.0614, 19.9383], zoom_start=11, tiles="cartodbpositron")

# Tworzymy paletę kolorów od zielonego do czerwonego (tani → drogi)
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap

price_min = top_expensive["price_m2"].min()
price_max = top_expensive["price_m2"].max()
norm = Normalize(vmin=price_min, vmax=price_max)
cmap = get_cmap("RdYlGn_r")

for _, row in top_expensive.iterrows():
    # Koordynaty: centroid dzielnicy jako fallback
    if row["district"] in centroids_dict:
        geom = centroids_dict[row["district"]]
        lat, lng = geom.y, geom.x
    else:
        continue
    
    # Kolor na podstawie ceny za m²
    color_normalized = norm(row["price_m2"])
    rgba = cmap(color_normalized)
    color_hex = "#{:02x}{:02x}{:02x}".format(
        int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255)
    )
    
    folium.CircleMarker(
        location=[lat, lng],
        radius=max(4, min(12, row["price_m2"] / 100)),
        color=color_hex,
        fill=True,
        fill_opacity=0.7,
        popup=(
            f"<b>{row['title'][:50]}</b><br>"
            f"Dzielnica: {row['district']}<br>"
            f"Cena: {row['price']:,.0f} PLN<br>"
            f"Powierzchnia: {row['surface']:.0f} m²<br>"
            f"<b>Cena za m²: {row['price_m2']:,.0f} PLN</b>"
        ),
        weight=2,
    ).add_to(m_expensive)

m_expensive


In [ ]:
# Styl: Python Graph Gallery - Circular Barplot (mediana ceny po dzielnicach)
circular_data = (
    krk.groupby("district", dropna=False)["price"]
    .median()
    .sort_values(ascending=False)
    .reset_index(name="median_price")
)

N = len(circular_data)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values = circular_data["median_price"].tolist()
labels = circular_data["district"].tolist()

# Zamykamy okrąg
angles += angles[:1]
values += values[:1]

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection="polar"))
bars = ax.bar(angles[:-1], values[:-1], width=0.5, alpha=0.8, color=sns.color_palette("RdYlGn_r", N))

# Stylizacja
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=9)
ax.set_ylim(0, max(values) * 1.1)
ax.set_title("Circular Barplot: mediana ceny ofert w Krakowie wg dzielnic", 
             fontsize=14, weight="bold", pad=20)
ax.grid(True, alpha=0.3)
plt.show()

# Styl: Python Graph Gallery - Bubble Chart (udział dzielnic w liczbie ofert)
bubble_data = (
    krk.groupby("district", dropna=False)
    .agg({
        "title": "count",
        "price": "median"
    })
    .reset_index()
    .rename(columns={"title": "listings"})
    .sort_values("listings", ascending=False)
)

# Proporcje dla bubble chart
bubble_data["bubble_size"] = (bubble_data["listings"] / bubble_data["listings"].sum()) * 10000

fig, ax = plt.subplots(figsize=(14, 10))

colors = sns.color_palette("viridis", len(bubble_data))
scatter = ax.scatter(
    x=range(len(bubble_data)),
    y=bubble_data["price"],
    s=bubble_data["bubble_size"],
    c=bubble_data["listings"],
    cmap="YlOrRd",
    alpha=0.6,
    edgecolors="black",
    linewidth=2
)

ax.set_xticks(range(len(bubble_data)))
ax.set_xticklabels(bubble_data["district"], rotation=45, ha="right")
ax.set_ylabel("Mediana ceny [PLN]", fontsize=11)
ax.set_xlabel("Dzielnica", fontsize=11)
ax.set_title("Bubble Chart: udział dzielnic (rozmiar bubble) vs mediana ceny", fontsize=14, weight="bold")

# Dodaj liczby ofert na bubblach
for idx, row in bubble_data.iterrows():
    ax.text(idx, row["price"], f"{int(row['listings'])}", 
            ha="center", va="center", fontsize=9, weight="bold", color="white")

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Liczba ofert", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Top oferty i eksport wyników
summary = (
    krk.groupby(["district", "subdistrict"], dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_surface=("surface", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
    .sort_values(["listings", "median_price"], ascending=[False, False])
)

print("Top 20 dzielnica/poddzielnica po liczbie ofert:")
display(summary.head(20))

out_path = Path("/content/krakow_offers_enriched.csv")
krk.to_csv(out_path, index=False)
print(f"Zapisano dane po czyszczeniu: {out_path}")